## **ML Process - Air Quality**
---
3 - Data Preprocessing

In [1]:
# Import the required libraries.
import yaml
import joblib
import numpy as np
import pandas as pd

from sklearn.preprocessing import (
    OneHotEncoder,
    LabelEncoder,
    StandardScaler
)

from imblearn.under_sampling import RandomUnderSampler as RUS
from imblearn.over_sampling import (
    RandomOverSampler as ROS,
    SMOTE
)

## **1. Configuration File**
---

In [2]:
# Function to load configuration parameter.
def load_config(config_path):
    """
    Load the configuration file (config.yaml).

    Parameters:
    ----------
    path_config : str
        Configuration file location.

    Returns:
    -------
    params : dict
        The configuration parameters.
    """

    # Try to load config.yaml file.
    try:
        with open(config_path, 'r') as file:
            params = yaml.safe_load(file)
    except FileNotFoundError as err:
        raise RuntimeError(f"Configuration file not found in {config_path}")

    return params

In [3]:
# Function to update configuration parameter.
def update_config(key, value, params, config_path):
    """
    Update the configuration parameter values.

    Parameters:
    ----------
    key : str
        The key to be updated.

    value : any type supported in Python
        The updated value.

    params : dict
        Loaded configuration parameters.

    path_config : str
        Configuration file location.

    Returns:
    -------
    config : dict
        Updated configuration parameters.
    """

    # To maintain the raw config immutable.
    params = params.copy()

    # Update the configuration parameters.
    params[key] = value

    with open(config_path, 'w') as file:
        yaml.dump(params, file)

    print(f"Params Updated! \nKey: {key} \nValue: {value}\n")

    # Reload the updated configuration parameters.
    config = load_config(config_path)

    return config

In [4]:
# Load the configuration file.
PATH_CONFIG = "../config/config.yaml"
config = load_config(PATH_CONFIG)

In [5]:
# Check the configuration parameters.
config

{'columns_datetime': ['tanggal'],
 'columns_int': ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max'],
 'features': ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2'],
 'label': 'category',
 'label_categories': ['BAIK', 'SEDANG', 'TIDAK SEHAT'],
 'label_categories_new': ['BAIK', 'TIDAK BAIK'],
 'object_columns': ['stasiun', 'critical', 'category'],
 'path_joined_data': '../data/interim/joined_dataset.pkl',
 'path_raw_data': '../data/raw/',
 'path_test_set': ['../data/interim/X_test.pkl', '../data/interim/y_test.pkl'],
 'path_train_set': ['../data/interim/X_train.pkl',
  '../data/interim/y_train.pkl'],
 'path_valid_set': ['../data/interim/X_valid.pkl',
  '../data/interim/y_valid.pkl'],
 'path_validated_data': '../data/interim/validated_data.pkl',
 'range_co': [-1, 47],
 'range_no2': [-1, 65],
 'range_o3': [-1, 151],
 'range_pm10': [-1, 179],
 'range_pm25': [-1, 174],
 'range_so2': [-1, 82],
 'range_stasiun': ['DKI1 (Bunderan HI)',
  'DKI2 (Kelapa Gading)',
  'DKI3 (Jagakarsa)',
  'DKI4

## **2. Load Data**
---

In [6]:
# Function for load data.
def load_data(config):
    """
    Load every set of data.

    Parameters:
    ----------
    config : dict
        The loaded configuration file.

    Returns:
    -------
    data_train, data_valid, data_test : pd.DataFrame
        The loaded data.
    """

    # Load the train set.
    X_train = joblib.load(config["path_train_set"][0])
    y_train = joblib.load(config["path_train_set"][1])

    # Load the valid set.
    X_valid = joblib.load(config["path_valid_set"][0])
    y_valid = joblib.load(config["path_valid_set"][1])

    # Load the test set.
    X_test = joblib.load(config["path_test_set"][0])
    y_test = joblib.load(config["path_test_set"][1])

    # Concatenate the X and y of each set.
    data_train = pd.concat([X_train, y_train], axis=1)
    data_valid = pd.concat([X_valid, y_valid], axis=1)
    data_test = pd.concat([X_test, y_test], axis=1)

    # Validate the proportion.
    num_all_data = int(data_train.shape[0]) + int(data_valid.shape[0]) + int(data_test.shape[0])
    print(f"Data train proportion : {len(X_train) / num_all_data}")
    print(f"Data valid proportion : {len(X_valid) / num_all_data}")
    print(f"Data test proportion  : {len(X_test) / num_all_data}")

    return data_train, data_valid, data_test

In [7]:
# Load the data.
data_train, data_valid, data_test = load_data(config)

Data train proportion : 0.7997793712079426
Data valid proportion : 0.09983452840595698
Data test proportion  : 0.10038610038610038


## **3. Join Categories**
---
SEDANG + TIDAK SEHAT => TIDAK BAIK

In [8]:
# Function for join categories.
def join_categories(data, config):
    """
    Join categories SEDANG & TIDAK SEHAT -> TIDAK BAIK.

    Parameters:
    ----------
    data : pd.DataFrame
        The loaded data.

    config : dict
        The loaded configuration file.

    Returns:
    -------
    data : pd.DataFrame
        The loaded data with categories joined.
    """

    # Check if label found in data.
    if config["label"] in data.columns.tolist():
        # Ensure raw data immutable.
        data = data.copy()

        # Rename SEDANG to TIDAK SEHAT.
        data["category"] = data["category"].replace("SEDANG", "TIDAK SEHAT")

        # Rename TIDAK SEHAT to TIDAK BAIK.
        data["category"] = data["category"].replace("TIDAK SEHAT", "TIDAK BAIK")

        return data
    else:
        raise RuntimeError("Label is not detected in the dataset.")

In [9]:
# Update the configuration parameter.
config = update_config(
    key = "label_categories_new",
    value = ["BAIK", "TIDAK BAIK"],
    params = config,
    config_path = PATH_CONFIG
)

Params Updated! 
Key: label_categories_new 
Value: ['BAIK', 'TIDAK BAIK']



In [10]:
data_train["category"].value_counts()

category
SEDANG         1044
TIDAK SEHAT     255
BAIK            151
Name: count, dtype: int64

In [11]:
data_train = join_categories(data_train, config)

In [12]:
data_train["category"].value_counts()

category
TIDAK BAIK    1299
BAIK           151
Name: count, dtype: int64

Join categories in valid data

In [13]:
data_valid["category"].value_counts()

category
SEDANG         130
TIDAK SEHAT     32
BAIK            19
Name: count, dtype: int64

In [14]:
data_valid = join_categories(data_valid, config)

In [15]:
data_valid["category"].value_counts()

category
TIDAK BAIK    162
BAIK           19
Name: count, dtype: int64

Join categories in test data

In [16]:
data_test["category"].value_counts()

category
SEDANG         131
TIDAK SEHAT     32
BAIK            19
Name: count, dtype: int64

In [17]:
data_test = join_categories(data_test, config)

In [18]:
data_test["category"].value_counts()

category
TIDAK BAIK    163
BAIK           19
Name: count, dtype: int64

## **4. Handling Missing Value**
---
- Create the nan_replace() function.

In [21]:
# Function to replace -1 with NaN.
def nan_replace(set_data):
    """
    Replace any -1 with NaN (Not a Number).

    Parameters:
    ----------
    data : pd.DataFrame
        The loaded data.

    Returns:
    -------
    data : pd.DataFrame
        The processed data.
    """

    # Ensure the raw data immutable.
    set_data = set_data.copy()

    set_data = set_data.replace(-1, np.nan)
    return set_data

In [22]:
data_train = nan_replace(data_train)
data_train.isnull().sum()

stasiun      0
pm10        45
pm25        62
so2         71
co          12
o3          39
no2         15
category     0
dtype: int64

In [23]:
data_valid = nan_replace(data_valid)
data_valid.isnull().sum()

stasiun      0
pm10         3
pm25         8
so2         14
co           2
o3           6
no2          2
category     0
dtype: int64

In [24]:
data_test = nan_replace(data_test)
data_test.isnull().sum()

stasiun      0
pm10         5
pm25        16
so2         12
co           2
o3           3
no2          2
category     0
dtype: int64

## **4.1 pm10 Imputation**
---
- Create the calculate_class_mean() and impute_class_mean() function.

In [25]:
# Function to calculate class mean for pm10 and pm25.
def calculate_class_mean(data, column):
    """
    Calculate the class mean for column pm10 and pm25.

    Parameters:
    ----------
    data : pd.DataFrame
        The loaded data.

    column : str
        The column name.

    Returns:
    -------
    impute_baik, impute_tidak_baik : float
        The mean for each class.
    """

    # Ensure raw data immutable.
    data = data.copy()

    # Boolean condition for each class.
    data_baik = data["category"] == "BAIK"
    data_tidak_baik = data["category"] == "TIDAK BAIK"

    # Calculate the class mean.
    impute_baik = float(data[data_baik][column].mean())
    impute_tidak_baik = float(data[data_tidak_baik][column].mean())

    print(f"Mean {column} class BAIK       : {impute_baik}")
    print(f"Mean {column} class TIDAK BAIK : {impute_tidak_baik}\n")    

    return impute_baik, impute_tidak_baik

In [26]:
# Function to impute missing values in column pm10 and pm25 using class mean.
def impute_class_mean(data, column, impute_baik, impute_tidak_baik):
    """
    Impute the missing value for column pm10 and pm25.

    Parameters:
    ----------
    data : pd.DataFrame
        The loaded data.

    column : str
        The column name.

    impute_baik : float
        The mean for class BAIK.

    impute_tidak_baik : float
        The mean for class TIDAK BAIK.
    
    Returns:
    -------
    data : pd.DataFrame
        The imputed data.
    """

    # Ensure raw data immutable.
    data = data.copy()

    # Boolean condition for each class.
    data_baik = data["category"] == "BAIK"
    data_tidak_baik = data["category"] == "TIDAK BAIK"

    # Boolean condition for missing values.
    missing_values = data[column].isnull() == True

    # Slice the missing values for each class.
    missing_baik = data[data_baik & missing_values]
    missing_tidak_baik = data[data_tidak_baik & missing_values]

    print(f"Num of missing value in {column} class BAIK before imputation       : {len(missing_baik)}")
    print(f"Num of missing value in {column} class TIDAK BAIK before imputation : {len(missing_tidak_baik)}\n")

    # Impute the missing values.
    data.loc[data[data_baik & missing_values].index, column] = impute_baik
    data.loc[data[data_tidak_baik & missing_values].index, column] = impute_tidak_baik

    print(f"Num of missing value in {column} class BAIK after imputation        : {data[data_baik][column].isnull().sum()}")
    print(f"Num of missing value in {column} class TIDAK BAIK after imputation  : {data[data_tidak_baik][column].isnull().sum()}\n")

    return data

In [28]:
# Calculate the class mean.
column = "pm10"

impute_baik, impute_tidak_baik = calculate_class_mean(
    data = data_train,
    column = column
)

# Update the configuration parameter.
config = update_config(
    key = f"impute_{column}",
    value = {"BAIK": impute_baik,
             "TIDAK BAIK": impute_tidak_baik},
    params = config,
    config_path = PATH_CONFIG
)

# Impute the missing values.
data_train = impute_class_mean(
    data = data_train,
    column = column,
    impute_baik = impute_baik,
    impute_tidak_baik = impute_tidak_baik
)

Mean pm10 class BAIK       : 28.54861111111111
Mean pm10 class TIDAK BAIK : 55.284694686756545

Params Updated! 
Key: impute_pm10 
Value: {'BAIK': 28.54861111111111, 'TIDAK BAIK': 55.284694686756545}

Num of missing value in pm10 class BAIK before imputation       : 7
Num of missing value in pm10 class TIDAK BAIK before imputation : 38

Num of missing value in pm10 class BAIK after imputation        : 0
Num of missing value in pm10 class TIDAK BAIK after imputation  : 0



In [29]:
# Impute the missing values.
data_valid = impute_class_mean(
    data = data_valid,
    column = column,
    impute_baik = impute_baik,
    impute_tidak_baik = impute_tidak_baik
)

Num of missing value in pm10 class BAIK before imputation       : 2
Num of missing value in pm10 class TIDAK BAIK before imputation : 1

Num of missing value in pm10 class BAIK after imputation        : 0
Num of missing value in pm10 class TIDAK BAIK after imputation  : 0



In [30]:
# Impute the missing values.
data_test = impute_class_mean(
    data = data_test,
    column = column,
    impute_baik = impute_baik,
    impute_tidak_baik = impute_tidak_baik
)

Num of missing value in pm10 class BAIK before imputation       : 1
Num of missing value in pm10 class TIDAK BAIK before imputation : 4

Num of missing value in pm10 class BAIK after imputation        : 0
Num of missing value in pm10 class TIDAK BAIK after imputation  : 0



## **4.2. pm25 Imputation**
---
Impute the pm25 column in train, valid, and test set

In [32]:
# Calculate the class mean.
column = "pm25"

impute_baik, impute_tidak_baik = calculate_class_mean(
    data = data_train,
    column = column
)

# Update the configuration parameter.
config = update_config(
    key = f"impute_{column}",
    value = {"BAIK": impute_baik,
             "TIDAK BAIK": impute_tidak_baik},
    params = config,
    config_path = PATH_CONFIG
)

# Impute the missing values.
data_train = impute_class_mean(
    data = data_train,
    column = column,
    impute_baik = impute_baik,
    impute_tidak_baik = impute_tidak_baik
)

Mean pm25 class BAIK       : 40.07692307692308
Mean pm25 class TIDAK BAIK : 82.52950432730134

Params Updated! 
Key: impute_pm25 
Value: {'BAIK': 40.07692307692308, 'TIDAK BAIK': 82.52950432730134}

Num of missing value in pm25 class BAIK before imputation       : 34
Num of missing value in pm25 class TIDAK BAIK before imputation : 28

Num of missing value in pm25 class BAIK after imputation        : 0
Num of missing value in pm25 class TIDAK BAIK after imputation  : 0



In [33]:
# Impute the missing values.
data_valid = impute_class_mean(
    data = data_valid,
    column = column,
    impute_baik = impute_baik,
    impute_tidak_baik = impute_tidak_baik
)

Num of missing value in pm25 class BAIK before imputation       : 4
Num of missing value in pm25 class TIDAK BAIK before imputation : 4

Num of missing value in pm25 class BAIK after imputation        : 0
Num of missing value in pm25 class TIDAK BAIK after imputation  : 0



In [34]:
# Impute the missing values.
data_test = impute_class_mean(
    data = data_test,
    column = column,
    impute_baik = impute_baik,
    impute_tidak_baik = impute_tidak_baik
)

Num of missing value in pm25 class BAIK before imputation       : 12
Num of missing value in pm25 class TIDAK BAIK before imputation : 4

Num of missing value in pm25 class BAIK after imputation        : 0
Num of missing value in pm25 class TIDAK BAIK after imputation  : 0



## **4.3. so2, co, o3, and no2 Imputation**
---
- Create the calculate_impute_values() and impute_missing_values() function.

In [35]:
# Function to calculate impute values for the other columns.
def calculate_impute_values(data):
    """
    Calculate the impute values for column so2, co, o3, and no2.
        - so2 imputed using the mean.
        - co, o3, and no2 imputed using the median.

    Parameters:
    ----------
    data : pd.DataFrame
        The loaded data.

    Returns:
    -------
    impute_values : dict
        The calculated impute values.
    """

    # Ensure raw data immutable.
    data = data.copy()

    # Calculate the impute values.
    impute_so2 = float(data["so2"].mean())
    impute_co = float(data["co"].median())
    impute_o3 = float(data["o3"].median())
    impute_no2 = float(data["no2"].median())

    impute_values = {
        "so2": impute_so2,
        "co": impute_co,
        "o3": impute_o3,
        "no2": impute_no2
    }

    return impute_values

In [36]:
# Function to impute missing values for the other columns.
def impute_missing_values(data, impute_values):
    """
    Impute the missing values for column so2, co, o3, and no2.

    Parameters:
    ----------
    data : pd.DataFrame
        The loaded data.

    impute_values : dict
        The calculated impute values.

    Returns:
    -------
    data : pd.DataFrame
        The imputed data.
    """

    # Ensure raw data immutable.
    data = data.copy()
    print(f"Num of missing values before imputation :\n{data.isnull().sum()}\n")
    
    # Impute the missing values.
    data = data.fillna(value = impute_values)
    print(f"Num of missing values after imputation  :\n{data.isnull().sum()}")

    return data

Impute the the other columns in train, valid, and test set.


In [38]:
# Calculate the impute values.
impute_values = calculate_impute_values(data_train)

# Update the configuration parameter.
cols = ['so2', 'co', 'o3', 'no2']
param_keys = ['impute_so2', 'impute_co', 'impute_o3', 'impute_no2']

for col, param_key in zip(cols, param_keys):
    config = update_config(
        key = param_key,
        value = impute_values[col],
        params = config,
        config_path = PATH_CONFIG
    )

# Impute the missing values.
data_train = impute_missing_values(
    data = data_train,
    impute_values = impute_values
)

Params Updated! 
Key: impute_so2 
Value: 35.191443074691804

Params Updated! 
Key: impute_co 
Value: 11.0

Params Updated! 
Key: impute_o3 
Value: 28.0

Params Updated! 
Key: impute_no2 
Value: 18.0

Num of missing values before imputation :
stasiun      0
pm10         0
pm25         0
so2         71
co          12
o3          39
no2         15
category     0
dtype: int64

Num of missing values after imputation  :
stasiun     0
pm10        0
pm25        0
so2         0
co          0
o3          0
no2         0
category    0
dtype: int64


In [39]:
# Impute the missing values.
data_valid = impute_missing_values(
    data = data_valid,
    impute_values = impute_values
)

Num of missing values before imputation :
stasiun      0
pm10         0
pm25         0
so2         14
co           2
o3           6
no2          2
category     0
dtype: int64

Num of missing values after imputation  :
stasiun     0
pm10        0
pm25        0
so2         0
co          0
o3          0
no2         0
category    0
dtype: int64


In [40]:
# Impute the missing values.
data_test = impute_missing_values(
    data = data_test,
    impute_values = impute_values
)

Num of missing values before imputation :
stasiun      0
pm10         0
pm25         0
so2         12
co           2
o3           3
no2          2
category     0
dtype: int64

Num of missing values after imputation  :
stasiun     0
pm10        0
pm25        0
so2         0
co          0
o3          0
no2         0
category    0
dtype: int64


## **5 - Encoding stasiun**
---
- Create the fit_ohe_encoder() and transform_ohe_encoder() function.

In [41]:
# Function to fit the encoder.
# ini hanya digunakan untuk data training
def fit_ohe_encoder(data, path_ohe):
    """
    Fit the OHE encoder.
    
    Parameters:
    ----------
    data : pd.Series
        Categorical input data.

    path_ohe : str
        The OHE encoder location.
    
    Returns:
    -------
    ohe_encoder : sklearn.preprocessing.OneHotEncoder
        Fitted OHE encoder object.
    """

    # Sklearn only accepts 2D matrix, thus we need to reshape the data.
    # kenapa pakai .reshape(-1,1) sklearn butuh input 2D, artinya lakukan (semua baris jadi 1 kolom)
    col_stasiun = np.array(data).reshape(-1, 1)

    # Create the encoder object.
    # kenapa perlu (sparse_output=False) karena Default OHE menghasilkan sparse matrix (hemat memori).
    # Tapi kita ingin hasilnya langsung jadi array biasa supaya mudah jadi DataFrame.
    ohe_encoder = OneHotEncoder(sparse_output=False)

    # Fit the encoder.
    # Fit artinya :
        # Pelajari kategori apa saja yang ada
        # Simpan daftar kategori itu
    ohe_encoder.fit(col_stasiun)
    
    # Serialize the ohe_encoder.  
    # .dump fungsi agar data berubah menjadi .yaml dan bisa update
    joblib.dump(ohe_encoder, path_ohe)
    
    return ohe_encoder

In [42]:
# Function to encode the data.
# ini digunakan untuk data train, data valid dan data test
def transform_ohe_encoder(data, encoder):
    """
    Transform the categorical column using OHE encoder.
    
    Parameters:
    ----------
    data : pd.DataFrame
        Data to be transformed.
        
    encoder : sklearn.preprocessing.OneHotEncoder
        The fitted encoder.
        
    Returns:
    -------
    data : pd.DataFrame
        The concatenated data with OHE columns.
    """

    # Ensure raw data immutable.
    data = data.copy()

    # Sklearn only accepts 2D matrix, thus we need to reshape the data.
    column = "stasiun"
    X_stasiun = np.array(data[column]).reshape(-1, 1)

    # Encode the data.
    # kenapa tidak pakai .fit, karena encoder sudah tahu kategori dari training.
    stasiun_features = encoder.transform(X_stasiun)

    # Convert to dataframe.
    # Kenapa dibuat dataframe lg ? supaya : Ada nama kolom, Index tetap sama, Mudah digabung dengan data lain.
    stasiun_features = pd.DataFrame(
        stasiun_features.tolist(),
        columns = list(encoder.categories_[0]),
        index = data.index
    )

    # Concat the OHE features with the original data.
    # ini digunakan untuk menggabungkan kolom hasil encoding dengan data asli.
    data = pd.concat(
        [stasiun_features, data],
        axis = 1
    )
    
    # Drop the original column. Kenapa di drop ?
    # Karena kolom "stasiun" sudah diganti dengan versi encoded
    data = data.drop(columns = column)

    # Convert columns type to string.
    new_col = [str(col_name) for col_name in data.columns.tolist()]
    data.columns = new_col
    
    return data

In [44]:
# Fit the ohe_encoder.
PATH_ENCODER_STASIUN = "../models/ohe_stasiun.pkl"

ohe_stasiun = fit_ohe_encoder(
    data = config["range_stasiun"],
    path_ohe = PATH_ENCODER_STASIUN
)

# Update the configuration parameter.
config = update_config(
    key = "path_fitted_encoder_stasiun",
    value = PATH_ENCODER_STASIUN,
    params = config,
    config_path = PATH_CONFIG
)

Params Updated! 
Key: path_fitted_encoder_stasiun 
Value: ../models/ohe_stasiun.pkl



Encode the stasiun column in train, valid, and test set.

In [45]:
# Encode the categorical column.
data_train = transform_ohe_encoder(
    data = data_train,
    encoder = ohe_stasiun
)

data_train.head()

,DKI1 (Bunderan HI),DKI2 (Kelapa Gading),DKI3 (Jagakarsa),DKI4 (Lubang Buaya),DKI5 (Kebon Jeruk) Jakarta Barat,pm10,pm25,so2,co,o3,no2,category
253,0.0,0.0,0.0,1.0,0.0,32.000000,60.0,39.000000,7.0,22.0,17.0,TIDAK BAIK
1632,0.0,0.0,0.0,1.0,0.0,55.284695,96.0,45.000000,10.0,33.0,17.0,TIDAK BAIK
880,0.0,0.0,0.0,1.0,0.0,38.000000,62.0,41.000000,8.0,12.0,7.0,TIDAK BAIK
78,0.0,0.0,1.0,0.0,0.0,20.000000,28.0,41.000000,3.0,28.0,5.0,BAIK
1658,0.0,0.0,0.0,0.0,1.0,70.000000,107.0,35.191443,11.0,39.0,16.0,TIDAK BAIK


In [46]:
# Encode the categorical column.
data_valid = transform_ohe_encoder(
    data = data_valid,
    encoder = ohe_stasiun
)

data_valid.head()

,DKI1 (Bunderan HI),DKI2 (Kelapa Gading),DKI3 (Jagakarsa),DKI4 (Lubang Buaya),DKI5 (Kebon Jeruk) Jakarta Barat,pm10,pm25,so2,co,o3,no2,category
258,0.0,0.0,0.0,1.0,0.0,78.0,136.0,41.0,14.0,32.0,18.0,TIDAK BAIK
1186,0.0,0.0,0.0,1.0,0.0,74.0,140.0,36.0,11.0,32.0,21.0,TIDAK BAIK
511,0.0,1.0,0.0,0.0,0.0,60.0,87.0,51.0,10.0,29.0,21.0,TIDAK BAIK
719,0.0,0.0,0.0,1.0,0.0,55.0,87.0,42.0,27.0,25.0,27.0,TIDAK BAIK
413,0.0,0.0,0.0,1.0,0.0,54.0,92.0,37.0,11.0,25.0,15.0,TIDAK BAIK


In [47]:
# Encode the categorical column.
data_test = transform_ohe_encoder(
    data = data_test,
    encoder = ohe_stasiun
)

data_test.head()

,DKI1 (Bunderan HI),DKI2 (Kelapa Gading),DKI3 (Jagakarsa),DKI4 (Lubang Buaya),DKI5 (Kebon Jeruk) Jakarta Barat,pm10,pm25,so2,co,o3,no2,category
1307,0.0,0.0,1.0,0.0,0.0,68.0,103.0,26.0,22.0,30.0,14.0,TIDAK BAIK
1793,0.0,0.0,0.0,1.0,0.0,68.0,131.0,39.0,14.0,24.0,25.0,TIDAK BAIK
676,0.0,0.0,1.0,0.0,0.0,51.0,70.0,33.0,8.0,37.0,10.0,TIDAK BAIK
1185,0.0,0.0,0.0,1.0,0.0,75.0,150.0,37.0,12.0,32.0,22.0,TIDAK BAIK
1634,0.0,0.0,0.0,1.0,0.0,71.0,120.0,45.0,13.0,36.0,21.0,TIDAK BAIK


## **6 - Scaling Data**
---
- Create the fit_scaler() and transform_scaler() function.

In [48]:
# Function to fit the scaler.
def fit_scaler(data, path_scaler, config):
    """
    Fit the scaler.
    
    Parameters:
    ----------
    data : pd.DataFrame
        Input data (all features must be in numeric form)

    path_scaler : str
        The scaler location.

    config : dict
        The loaded configuration file.
        
    Returns:
    -------
    scaler : sklearn.preprocessing.StandardScaler
        Fitted scaler object (storing the mean & std of all features)
    """

    # Split input-output, StandardScaler() only accepts numeric data.
    label = config["label"]
    y = data[label]
    X = data.drop(columns = label)

    # Create scaler object.
    scaler = StandardScaler()

    # Fit the scaler.
    scaler.fit(X)

    # Serialize the scaler.    
    joblib.dump(scaler, path_scaler)
    
    return scaler

In [49]:
# Function to scale the data.
def transform_scaler(data, scaler, config):
    """
    Transform the data using scaler.
    
    Parameters:
    ----------
    data : pd.DataFrame
        Input data (all features must be in numeric form)    
        
    scaler : sklearn.preprocessing.StandardScaler
        Fitted scaler object (storing the mean & std of all features)

    config : dict
        The loaded configuration file.
        
    Returns:
    -------
    data : pd.DataFrame
        The scaled data
    """

    # Ensure raw data immutable.
    data = data.copy()

    # Split input-output, StandardScaler() only accepts numeric data.
    label = config["label"]
    y = data[label]
    X = data.drop(columns = label)

    # Scale the data.
    scaled_data = scaler.transform(X)

    # Convert to dataframe.
    X_scaled = pd.DataFrame(
        scaled_data,
        columns = X.columns,
        index = X.index
    )

    # Concat the X_scaled with y.
    data = pd.concat(
        [X_scaled, y],
        axis = 1
    )
    
    return data

In [51]:
# Fit the scaler.
PATH_SCALER = "../models/scaler.pkl"

scaler = fit_scaler(
    data = data_train,
    path_scaler = PATH_SCALER,
    config = config
)

# Update the configuration parameter.
config = update_config(
    key = "path_fitted_scaler",
    value = PATH_SCALER,
    params = config,
    config_path = PATH_CONFIG
)

Params Updated! 
Key: path_fitted_scaler 
Value: ../models/scaler.pkl



Scale the data in train, valid, and test set.

In [52]:
# Scale the data.
data_train = transform_scaler(
    data = data_train,
    scaler = scaler,
    config = config
)

print(f"Data shape : {data_train.shape}")
data_train.head()

Data shape : (1450, 12)


,DKI1 (Bunderan HI),DKI2 (Kelapa Gading),DKI3 (Jagakarsa),DKI4 (Lubang Buaya),DKI5 (Kebon Jeruk) Jakarta Barat,pm10,pm25,so2,co,o3,no2,category
253,-0.513972,-0.494606,-0.5,1.995700,-0.490282,-1.404055,-0.739893,0.311896,-0.953887,-0.662109,-0.270404,TIDAK BAIK
1632,-0.513972,-0.494606,-0.5,1.995700,-0.490282,0.190690,0.731020,0.803256,-0.337699,0.094655,-0.270404,TIDAK BAIK
880,-0.513972,-0.494606,-0.5,1.995700,-0.490282,-0.993121,-0.658175,0.475682,-0.748491,-1.350077,-1.364700,TIDAK BAIK
78,-0.513972,-0.494606,2.0,-0.501077,-0.490282,-2.225923,-2.047371,0.475682,-1.775471,-0.249329,-1.583559,BAIK
1658,-0.513972,-0.494606,-0.5,-0.501077,2.039643,1.198526,1.180465,0.000000,-0.132303,0.507435,-0.379834,TIDAK BAIK


In [53]:
# Scale the data.
data_valid = transform_scaler(
    data = data_valid,
    scaler = scaler,
    config = config
)

print(f"Data shape : {data_valid.shape}")
data_valid.head()

Data shape : (181, 12)


,DKI1 (Bunderan HI),DKI2 (Kelapa Gading),DKI3 (Jagakarsa),DKI4 (Lubang Buaya),DKI5 (Kebon Jeruk) Jakarta Barat,pm10,pm25,so2,co,o3,no2,category
258,-0.513972,-0.494606,-0.5,1.995700,-0.490282,1.746438,2.365367,0.475682,0.483885,0.025858,-0.160975,TIDAK BAIK
1186,-0.513972,-0.494606,-0.5,1.995700,-0.490282,1.472482,2.528802,0.066215,-0.132303,0.025858,0.167314,TIDAK BAIK
511,-0.513972,2.021811,-0.5,-0.501077,-0.490282,0.513636,0.363292,1.294616,-0.337699,-0.180532,0.167314,TIDAK BAIK
719,-0.513972,-0.494606,-0.5,1.995700,-0.490282,0.171191,0.363292,0.557576,3.154032,-0.455719,0.823892,TIDAK BAIK
413,-0.513972,-0.494606,-0.5,1.995700,-0.490282,0.102702,0.567585,0.148109,-0.132303,-0.455719,-0.489263,TIDAK BAIK


In [54]:
# Scale the data.
data_test = transform_scaler(
    data = data_test,
    scaler = scaler,
    config = config
)

print(f"Data shape : {data_test.shape}")
data_test.head()

Data shape : (182, 12)


,DKI1 (Bunderan HI),DKI2 (Kelapa Gading),DKI3 (Jagakarsa),DKI4 (Lubang Buaya),DKI5 (Kebon Jeruk) Jakarta Barat,pm10,pm25,so2,co,o3,no2,category
1307,-0.513972,-0.494606,2.0,-0.501077,-0.490282,1.061548,1.017031,-0.752718,2.127052,-0.111735,-0.598693,TIDAK BAIK
1793,-0.513972,-0.494606,-0.5,1.995700,-0.490282,1.061548,2.161074,0.311896,0.483885,-0.524516,0.605032,TIDAK BAIK
676,-0.513972,-0.494606,2.0,-0.501077,-0.490282,-0.102765,-0.331306,-0.179465,-0.748491,0.369842,-1.036411,TIDAK BAIK
1185,-0.513972,-0.494606,-0.5,1.995700,-0.490282,1.540971,2.937389,0.148109,0.073093,0.025858,0.276744,TIDAK BAIK
1634,-0.513972,-0.494606,-0.5,1.995700,-0.490282,1.267015,1.711628,0.803256,0.278489,0.301045,0.167314,TIDAK BAIK


## **7 - Balancing Label**
---


In [55]:
# Check the label distribution.
data_train["category"].value_counts(normalize=True)

category
TIDAK BAIK    0.895862
BAIK          0.104138
Name: proportion, dtype: float64

Seems like the label is highly imbalanced. We need to balancing the label.

In [56]:
# Function to balancing the label.
def label_balancer(data, balancer_type, config, random_state=123):
    """
    Balancing the category label.

    Parameters:
    ----------
    data : pd.DataFrame
        The scaled data.

    balancer_type : str
        The balancer type.

    config : dict
        The loaded configuration file.

    random_state : int, default = 123
        For reproducibility.

    Returns:
    -------
    X_balanced : pd.DataFrame
        The features with balanced label.

    y_balanced : pd.Series
        The label with balanced label.
    """

    # Ensure the raw data immutable.
    data = data.copy()

    # Split input-output, imblearn-style similar to sklearn-style.
    label = config["label"]
    y = data[label]
    X = data.drop(columns = label)

    # Set the balancer.
    list_balancer = ["rus", "ros", "sm"]

    if str(balancer_type).lower() not in list_balancer:
        raise RuntimeError("The balancer type is invalid.")
    else:
        if str(balancer_type).lower() == "rus":
            balancer = RUS(random_state = random_state)            
        elif str(balancer_type).lower() == "ros":
            balancer = ROS(random_state = random_state)
        else:
            balancer = SMOTE(random_state = random_state)

        # Fit resample the balancer.
        X_balanced, y_balanced = balancer.fit_resample(X, y)

        print(f"The label are balanced using {balancer.__class__.__name__}")

        # Check the label distribution.
        print(y_balanced.value_counts())

        return X_balanced, y_balanced

In [57]:
# Label balancing.
X_rus, y_rus = label_balancer(
    data = data_train,
    balancer_type = "rus",
    config = config
)

The label are balanced using RandomUnderSampler
category
BAIK          151
TIDAK BAIK    151
Name: count, dtype: int64


In [58]:
# Label balancing.
X_ros, y_ros = label_balancer(
    data = data_train,
    balancer_type = "ros",
    config = config
)

The label are balanced using RandomOverSampler
category
TIDAK BAIK    1299
BAIK          1299
Name: count, dtype: int64


In [59]:
# Label balancing.
X_sm, y_sm = label_balancer(
    data = data_train,
    balancer_type = "sm",
    config = config
)

The label are balanced using SMOTE
category
TIDAK BAIK    1299
BAIK          1299
Name: count, dtype: int64


## **8 - Label Encoding**
---
- Create the fit_label_encoder() and transform_label_encoder() function.

In [60]:
# Function to fit label encoder.
def fit_label_encoder(label, path_le):
    """
    Fit the label encoder.

    Parameters:
    ----------
    label : pd.Series
        Categorical label.

    path_le : str
        The label encoder location.

    Returns:
    -------
    label_encoder : sklearn.preprocessing.LabelEncoder
        Fitted label encoder object.
    """

    # Create the label encoder object.
    label_encoder = LabelEncoder()

    # Fit the label encoder.
    label_encoder.fit(label)

    # Serialize the label encoder.
    joblib.dump(label_encoder, path_le)

    return label_encoder

In [61]:
# Function to encode the label.
def transform_label_encoder(label, encoder):
    """
    Transform the categorical label using label encoder.

    Parameters:
    ----------
    label : pd.Series
        Categorical label.

    encoder : sklearn.preprocessing.LabelEncoder
        Fitted label encoder object.

    Returns:
    encoded_label : pd.Series
        The encoded label.
    """

    # Ensure raw label immutable.
    label = label.copy()

    # Encode the label.
    encoded_label = pd.Series(
        encoder.transform(label)
    )

    return encoded_label

In [63]:
# Fit the label_encoder.
PATH_ENCODER_LABEL = "../models/label_encoder.pkl"

label_encoder = fit_label_encoder(
    label = config["label_categories_new"],
    path_le = PATH_ENCODER_LABEL
)

# Update the configuration parameter.
config = update_config(
    key = "path_fitted_encoder_label",
    value = PATH_ENCODER_LABEL,
    params = config,
    config_path = PATH_CONFIG
)

Params Updated! 
Key: path_fitted_encoder_label 
Value: ../models/label_encoder.pkl



In [64]:
# Encode the label for train set.
y_rus = transform_label_encoder(
    label = y_rus,
    encoder = label_encoder
)

y_ros = transform_label_encoder(
    label = y_ros,
    encoder = label_encoder
)

y_sm = transform_label_encoder(
    label = y_sm,
    encoder = label_encoder
)

In [65]:
# Encode the label for valid and test set.
label = config["label"]

y_valid = transform_label_encoder(
    label = data_valid[label],
    encoder = label_encoder
)
X_valid = data_valid.drop(columns = label)

y_test = transform_label_encoder(
    label = data_test[label],
    encoder = label_encoder
)
X_test = data_test.drop(columns = label)

## **9 - Data Serialization**
---

In [66]:
# Define data configuration.
X_train = {
    "Undersampling": X_rus,
    "Oversampling": X_ros,
    "SMOTE": X_sm
}

y_train = {
    "Undersampling": y_rus,
    "Oversampling": y_ros,
    "SMOTE": y_sm
}

data_configuration = {
    "train": {
        "X_train": X_train,
        "y_train": y_train
    },
    "valid": {
        "X_valid": X_valid,
        "y_valid": y_valid
    },
    "test": {
        "X_test": X_test,
        "y_test": y_test
    }
}

In [68]:
# Serialize the preprocessed data.
PATH_PROCESSED_DATA = "../data/processed/"

for key, value in data_configuration.items():
    config_key = f"path_clean_{key}"
    config_value = []

    for v in value:
        # Get each path.
        path = f"{PATH_PROCESSED_DATA + v}_clean.pkl"
        config_value.append(path)

        # Get each data.
        data = value[v]

        # Serialize the preprocessed data.
        joblib.dump(data, path)

    # Update the configuration parameters.
    config = update_config(
        key = config_key,
        value = config_value,
        params = config,
        config_path = PATH_CONFIG
    )

Params Updated! 
Key: path_clean_train 
Value: ['../data/processed/X_train_clean.pkl', '../data/processed/y_train_clean.pkl']

Params Updated! 
Key: path_clean_valid 
Value: ['../data/processed/X_valid_clean.pkl', '../data/processed/y_valid_clean.pkl']

Params Updated! 
Key: path_clean_test 
Value: ['../data/processed/X_test_clean.pkl', '../data/processed/y_test_clean.pkl']



In [69]:
# Check the configuration parameters.
config

{'columns_datetime': ['tanggal'],
 'columns_int': ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max'],
 'features': ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2'],
 'impute_co': 11.0,
 'impute_no2': 18.0,
 'impute_o3': 28.0,
 'impute_pm10': {'BAIK': 28.54861111111111, 'TIDAK BAIK': 55.284694686756545},
 'impute_pm25': {'BAIK': 40.07692307692308, 'TIDAK BAIK': 82.52950432730134},
 'impute_so2': 35.191443074691804,
 'label': 'category',
 'label_categories': ['BAIK', 'SEDANG', 'TIDAK SEHAT'],
 'label_categories_new': ['BAIK', 'TIDAK BAIK'],
 'object_columns': ['stasiun', 'critical', 'category'],
 'path_clean_test': ['../data/processed/X_test_clean.pkl',
  '../data/processed/y_test_clean.pkl'],
 'path_clean_train': ['../data/processed/X_train_clean.pkl',
  '../data/processed/y_train_clean.pkl'],
 'path_clean_valid': ['../data/processed/X_valid_clean.pkl',
  '../data/processed/y_valid_clean.pkl'],
 'path_fitted_encoder_label': '../models/label_encoder.pkl',
 'path_fitted_encoder_stasiu